In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
CRS = 'EPSG:4326'

### Data imports / exports

In [ ]:
# import paths
PATH_TO_ACLED_EVENTS = '../data/raw/Middle-East_aggregated_data_up_to_week_of-2026-10-31.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = '../data/processed/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = '../data/processed/acled/conflict_events_geolocated.csv'

# import data
df_events = pd.read_csv(PATH_TO_ACLED_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [ ]:
#df_events.head()

In [ ]:
#gdf_boundaries.head()

### Preprocessing

In [ ]:
# strip whitespaces
string_cols = ['REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
df_events[string_cols] = df_events[string_cols].apply(lambda x: x.str.strip())

# format date columns
df_events['WEEK'] = pd.to_datetime(df_events['WEEK'])

# create date grouping columns
df_events['YEAR'] = df_events['WEEK'].dt.year
df_events['YEARMO'] = df_events['WEEK'].dt.to_period('M').astype(str)
df_events['YEAR_WEEK'] = df_events['WEEK'].dt.to_period('W').astype(str)

mask_palestine_events = df_events.COUNTRY == 'Palestine'
df_events = df_events[mask_palestine_events]

### Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_events = gpd.GeoDataFrame(
    df_events,
    geometry=gpd.points_from_xy(
        df_events['CENTROID_LONGITUDE'],
        df_events['CENTROID_LATITUDE']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_events.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_events = gdf_events.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_events.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_events_joined = gpd.sjoin(
    gdf_events,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_events_joined))
print("Matched events:", gdf_events_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_events_joined['adm1_name'].isna().sum())

In [ ]:
unmatched = gdf_events_joined[gdf_events_joined['ADMIN1'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

In [ ]:
 gdf_events_joined['adm1_name'].value_counts()

In [ ]:
 gdf_events_joined['adm2_name'].value_counts()

#### TODO: Why are all events confined to one governorate per administrative boundary?

In [ ]:
# do all admin 1 names agree between sources?
assert(len(gdf_events_joined[gdf_events_joined.ADMIN1 != gdf_events_joined.adm1_name]) == 0)

In [ ]:
# Check whether any original event IDs appear more than once
print(gdf_events_joined.index.value_counts().value_counts().sort_index())

<br>

### Governorate Aggregates

#### Total Events (all)

In [ ]:
#gdf_events_joined.head()

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

#### Total Events (by type)

Across all years

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

<br>

Broken down by year, and ratio (normalized count) is for the year

This would allow us to see how specific event types are changing over time by governorate

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'YEAR', 'EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name', 'YEAR'])['count'].transform('sum')
).round(2)

event_counts
# print(event_counts.to_string()) # to see all rows

<br>

Broken down by event subtype (could also be interesting to see subtype broken down by year, but would be too large to comprehend upon manual visual inspection)

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'SUB_EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

<br>

#### Total Fatalities

Across all years and event types

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

Across all years, broken down by event type

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'EVENT_TYPE'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

Broken down by year, and ratio (normalized count) is for the year

This would allow us to see how specific event types are changing over time

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'YEAR', 'EVENT_TYPE'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name', 'YEAR'])['count'].transform('sum')
).round(2)

event_counts
#print(event_counts.to_string()) # to see all rows

### Export Results

In [ ]:
# geopackage for further geoprocessing and visualization
gdf_events_joined.to_file(
    PATH_TO_GEOLOCATED_EVENTS_GPKG,
    driver='GPKG'
)

# df for further analysis
df_final = pd.DataFrame(gdf_events_joined.drop(columns='geometry'))
df_final.to_csv(PATH_TO_GEOLOCATED_EVENTS_DF, index=False)